In [ ]:
'''
the cat sat on the mat
RNN 계열 : 왼쪽에서 오른쪽, 단점 : 멀리 떨어진 단어들은 서로 영향을 주고 받기 어려움

The  cat that the boy who lived here adopted is sleeping
cat vs. sleeping

self-attention
Q: Query 찾고 싶은 정보
K: Knowledge 가진 정보
V: 최종 전달할 정보

비교대상                    유사도           의미
cat vs. the                낮음             the 의미 없음
cat vs. cat                높음             자기자신
cat vs. sat                중간             동사와 연결되어 행위를 나타냄
cat vs. on                 낮음             on 전치사...
cat vs. mat                낮음             의미적으로 멀다

softmax로 중요도 확률처럼 변경(유사도를 가중치로 변환)
단어                        가중치    
the                         0.05
cat                         0.06
sat                         0.3
mat                         0.05

cat이 보는 시점은
Query(cat) -> compare with -> key(the) -> key(cat) -> key(sat)

가중치
the : 0.1    cat : 0.7     sat : 0.2
출력 : 0.1 * value(the) + ... 

RNN 순차처리(왼->오) 가능
self-attention은 병렬처리가 가능

다중의미처리
river bank  river를 강하게 강조
bank loan   loan을 강하게 참조
'''

In [ ]:
# self-attention 시각화
import numpy as np
import matplotlib.pyplot as plt

words = ['cat','fish','like']  # 고양이가 생선을 좋아한다
# 가상의 attention 가중치
# 각 행은 해당 단어가 다른단어들에게 주목하는 정도
attention_weight = np.array([
    [0.7,0.2,0.1],  # 고양이는 자가자신에게 가장 높은 가중치
    [0.3,0.5,0.2],  #
    [0.4,0.3,0.3]
])

fig, ax = plt.subplots(figsize=(10,8))
im = ax.imshow(attention_weight, cmap='YlOrRd')

ax.set_xticks(range(len(words)))
ax.set_yticks(range(len(words)))
ax.set_xticklabels(words)
ax.set_yticklabels(words)

ax.set_xlabel('key')
ax.set_ylabel('query')


In [ ]:
# Beam Search
# 문장을 생성할 때 다음에 나올 단어는 수천~수만개가 될 수 있는데, 이걸 경우의 수로 따지면 답이 없음
# 상위 N개의 후보만 유지, N을 beam size
# beam size = 1 매번 가장 좋은 것만 선택(Greedy)
# beam size = 4 4개의 가능성을 동시에 탐색

In [12]:
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM
import torch
import time
MODEL_NAME = 't5-small'
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
model = AutoModelForSeq2SeqLM.from_pretrained(MODEL_NAME)
device = 'cuda' if torch.cuda.is_available() else 'cpu'
model = model.to(device)

text = """summarize: The Amazon rainforest is the world's largest tropical rainforest.
  It covers much of northwestern Brazil and extends into Colombia, Peru and other South American countries.
  The Amazon is home to millions of species of plants and animals, many of which are found nowhere else on Earth.
  However, deforestation poses a significant threat to this vital ecosystem."""

print('원본 : ')
print(text.replace('summarize:',''))
inputs = tokenizer(text, return_tensors='pt',max_length=512,truncation=True).to(device)
# 다양한 beam size 실험
beam_sizes = [1,2,4,8]
results = []
for num_beams in beam_sizes:
  print(f'beam size : {num_beams}')
  start_time = time.time()
  outputs = model.generate(
      **inputs,
      num_beams = num_beams,
      max_length=60,
      min_length=20,
      early_stopping = True,
      no_repeat_ngram_size = 3,
      num_return_sequences=1
  )
  elapsed_time = time.time() - start_time
  summary = tokenizer.decode(outputs[0],skip_special_tokens=True)
  results.append((num_beams,summary,elapsed_time))

원본 : 
 The Amazon rainforest is the world's largest tropical rainforest.
  It covers much of northwestern Brazil and extends into Colombia, Peru and other South American countries.
  The Amazon is home to millions of species of plants and animals, many of which are found nowhere else on Earth.
  However, deforestation poses a significant threat to this vital ecosystem.
beam size : 1
beam size : 2
beam size : 4
beam size : 8


In [13]:
for num_beams,summary,elapsed_time in results:
  print(f'측정시간 : {elapsed_time}')
  print(f'beam size : {num_beams}')
  print(summary)
  print()

측정시간 : 0.8660888671875
beam size : 1
the amazon rainforest covers much of northwestern Brazil. it extends into Colombia, Peru and other South american countries.

측정시간 : 0.8932263851165771
beam size : 2
the amazon rainforest covers much of northwestern Brazil. it extends into Colombia, Peru and other south american countries.

측정시간 : 1.3758478164672852
beam size : 4
the amazon rainforest covers much of northwestern Brazil and extends into Colombia, Peru and other South American countries. deforestation poses a significant threat to this vital ecosystem.

측정시간 : 1.172224998474121
beam size : 8
the amazon rainforest covers much of northwestern Brazil. it extends into Colombia, Peru and other south american countries.



In [14]:
for num_beams, summary, elapsed_time in result:
    print(f'측정시간 : {elapsed_time}')
    print(f'beam size : {num_beams}')
    print(summary)
    print()

# Beam = 1(Greedy) 실시간 시스템
# Beam = 4~5 품질과 속도의 균형(가장 일반적)
# Beam = 8~10 최고 품질이 필요한 경우(논문, 공식문서)
# Beam 

NameError: name 'result' is not defined

In [ ]:
'''
ROUGE 메트릭
문서를 요약했는데 어떻게 품질을 측정
겹침정도를 측정
ROUGE-1 단어단위
    정답 : 고양이가 생선을 먹었다
    생성 : 고양이가 물고기를 먹었다
    겹침 : 고양이가, 먹었다, 2/4=0.5
ROUGE-2 2개 단어 단위 : 순서도 고려 why? 2개씩이니까
ROUGE-L : 가장 긴 공통 부분수열(순서는 유지하지만 연속적이지 않아도 됨)

단점 : 의미는 같지만 다른 표현을 쓰면 점수가 낮다
    자동차 vs 차량 겹침 없음으로 판단
'''

In [15]:
# 토크나이져

In [ ]:
# 배치처리, 데이터 콜레이터(배치를 만들때 길이를 맞춰주는 작업)
# 효율적인 데이터 처리방법
import torch,time
from transformers import AutoTokenizer, DataCollatorForSeq2Seq, AutoModelForSeq2SeqLM
# AutoModelForSeq2SeqLM : 입력을 받아서 다른 텍스트 생성하는 seq2seq모델을 자동로드
# Encoder-Decoder모델을 자동으로
# T5, BART, MarianMT
# 번역/요약/QA/문장변환  입력->출력

# DataCollatorForSeq2Seq
# seq2seq 학습시 배치단위로 패딩-정렬-라벨 시프트 등을 자동처리하는 데이터 정렬 도구
# 배치생성 - 길이다라느 문장들을 동일길이로 패딩
# 라벨 시프트 - 라벨을 디코더 입력으로 사용  (teacher forcing에필요한 작업)
# DataLoader 안에서 사용

MODEL_NAME = 't5-small'
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
model = AutoModelForSeq2SeqLM.from_pretrained(MODEL_NAME)

device = 'cuda' if torch.cuda.is_available() else 'cpu'
model = model.to(device)
model.eval()

texts = [
    "summarize: The cat sat on the mat.",
    "summarize: Python is a popular programming language.",
    "summarize: Machine learning is a subset of artificial intelligence.",
    "summarize: The weather is nice today.",
    "summarize: I love reading books in my free time.",
    "summarize: Coffee is one of the most popular beverages worldwide.",
    "summarize: Regular exercise is important for health.",
    "summarize: The Internet has changed how we communicate.",
]
# 개별처리 VS 배치처리
start_time = time.time()
result_indivisual = []
with torch.no_grad():
  for text in texts:
    inputs = tokenizer(text, return_tensors='pt',max_length=512,truncation=True).to(device)
    outputs = model.generate(**inputs,max_length=30)
    summary = tokenizer.decode(outputs[0],skip_special_tokens=True)
    result_indivisual.append(summary)
time_indivisual = time.time() - start_time
print(f'소요시간 : {time_indivisual}')
print(f'문장당: {time_indivisual / len(texts):.4f}')

In [ ]:
# 배치처리
start_time = time.time()
result_batch = []
outputs = model.generate(**inputs,max_length=30)
for output in outputs:
  summary = tokenizer.decode(output, skip_special_tokens=True)
  result_batch.append(summary)
time_indivisual = time.time() - start_time
print(f'소요시간 : {time_indivisual}')
print(f'문장당: {time_indivisual / len(texts):.4f}')

In [ ]:
# 데이터콜레이터
start_time = time.time()
data_collator =  DataCollatorForSeq2Seq(tokenizer=tokenizer,model=model, padding=True)
tokenized = []
for text in texts:
  encoded = tokenizer(text,truncation=True)
  tokenized.append(encoded)
batch = data_collator(tokenized)

time_indivisual = time.time() - start_time
print(f'소요시간 : {time_indivisual}')
print(f'문장당: {time_indivisual / len(texts):.4f}')
